<a href="https://colab.research.google.com/github/nbhargaveepriya-dev/SKILL_SWAP/blob/master/ai_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q mediapipe opencv-python-headless
# Download the specific "Blendshape" capable face model and the Object Detector
!wget -q -O face_landmarker.task https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task
!wget -q -O detector.tflite https://storage.googleapis.com/mediapipe-models/object_detector/efficientdet_lite0/float16/1/efficientdet_lite0.tflite

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 88.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 11.4 MB/s eta 0:00:00


In [ ]:
import cv2
import time
import numpy as np
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
from google.colab.output import eval_js
from base64 import b64decode
from IPython.display import display, Javascript

# --- 1. SETUP AI MODELS ---
base_face = python.BaseOptions(model_asset_path='face_landmarker.task')
options_face = vision.FaceLandmarkerOptions(base_options=base_face, output_face_blendshapes=True, num_faces=1)
face_landmarker = vision.FaceLandmarker.create_from_options(options_face)

base_obj = python.BaseOptions(model_asset_path='detector.tflite')
options_obj = vision.ObjectDetectorOptions(base_options=base_obj, score_threshold=0.3)
obj_detector = vision.ObjectDetector.create_from_options(options_obj)

# --- 2. THE JAVASCRIPT WEBCAM FIX ---
# This script creates a NEW video element that is visible in the output
def start_webcam():
  js = Javascript('''
    async function showVideo() {
      const div = document.createElement('div');
      div.style.padding = '10px';
      div.style.background = '#111';
      div.style.width = 'max-content';
      div.style.borderRadius = '10px';

      const title = document.createElement('h3');
      title.innerText = '🛡️ FOCUS GUARD: LIVE CAMERA';
      title.style.color = 'white';
      title.style.margin = '0 0 10px 0';
      div.appendChild(title);

      const statusBox = document.createElement('div');
      statusBox.id = 'status-display';
      statusBox.style.fontSize = '24px';
      statusBox.style.fontWeight = 'bold';
      statusBox.style.color = '#4CAF50';
      statusBox.innerText = 'STARTING...';
      div.appendChild(statusBox);

      const video = document.createElement('video');
      video.id = 'video-stream';
      video.style.display = 'block';
      video.style.borderRadius = '8px';
      video.width = 640;
      video.height = 480;
      video.autoplay = true;
      div.appendChild(video);

      document.body.appendChild(div);

      const stream = await navigator.mediaDevices.getUserMedia({video: true});
      video.srcObject = stream;
      await video.play();
    }
    showVideo();
  ''')
  display(js)

def capture_frame():
  data = eval_js('''
    (async () => {
      const video = document.getElementById('video-stream');
      const canvas = document.createElement('canvas');
      canvas.width = 640; canvas.height = 480;
      canvas.getContext('2d').drawImage(video, 0, 0);
      return canvas.toDataURL('image/jpeg', 0.8);
    })()
  ''')
  return data

def update_ui(msg, color):
  eval_js(f'document.getElementById("status-display").innerText = "{msg}";')
  eval_js(f'document.getElementById("status-display").style.color = "{color}";')

# --- 3. RUN THE APP ---
start_webcam()
time.sleep(2) # Wait for camera to warm up

study_goal = 60
current_focus = 0

print("Guard is active! Stay focused.")

try:
  while current_focus < study_goal:
    frame_data = capture_frame()
    img_bytes = b64decode(frame_data.split(',')[1])
    img = cv2.imdecode(np.frombuffer(img_bytes, dtype=np.uint8), cv2.IMREAD_COLOR)
    mp_img = mp.Image(image_format=mp.ImageFormat.SRGB, data=cv2.cvtColor(img, cv2.COLOR_BGR2RGB))

    # AI DETECTION
    face_res = face_landmarker.detect(mp_img)
    obj_res = obj_detector.detect(mp_img)

    # LOGIC
    phone = any(d.categories[0].category_name == 'cell phone' for d in obj_res.detections)
    eyes_closed = False

    if face_res.face_blendshapes:
      shapes = {b.category_name: b.score for b in face_res.face_blendshapes[0]}
      if shapes.get('eyeBlinkLeft', 0) > 0.4 and shapes.get('eyeBlinkRight', 0) > 0.4:
        eyes_closed = True

    # ACTION
    if phone:
        update_ui("📵 PHONE DETECTED!", "#ff4444")
    elif eyes_closed:
        update_ui("😴 WAKE UP!", "#ffbb00")
    elif not face_res.face_landmarks:
        update_ui("👤 WHERE ARE YOU?", "#ffbb00")
    else:
        current_focus += 1
        update_ui(f"✅ FOCUSING... {current_focus}/{study_goal}s", "#4CAF50")

    time.sleep(0.5)
except Exception as e:
    print(f"Stopping: {e}")

update_ui("🎉 SESSION COMPLETE!", "#00ff00")

<IPython.core.display.Javascript object>

Guard is active! Stay focused.


In [ ]:
import cv2
import time
import numpy as np
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
from google.colab.output import eval_js
from base64 import b64decode
from IPython.display import display, Javascript

# ==========================================
# 1. INITIALIZE AI MODELS (2026 API)
# ==========================================
base_face = python.BaseOptions(model_asset_path='face_landmarker.task')
options_face = vision.FaceLandmarkerOptions(
    base_options=base_face,
    output_face_blendshapes=True, # For Eye Detection
    num_faces=1
)
face_landmarker = vision.FaceLandmarker.create_from_options(options_face)

base_obj = python.BaseOptions(model_asset_path='detector.tflite')
# Lower threshold (0.25) makes phone detection "stricter"
options_obj = vision.ObjectDetectorOptions(base_options=base_obj, score_threshold=0.25)
obj_detector = vision.ObjectDetector.create_from_options(options_obj)

# ==========================================
# 2. DESIGN & WEBCAM UI (MIRRORED & AUDIO)
# ==========================================
def start_guard_ui():
    js = Javascript('''
        async function setup() {
            // 1. Create the UI Container
            const div = document.createElement('div');
            div.style = "background:#121212; color:#e0e0e0; padding:25px; border-radius:20px; width:660px; font-family:'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; box-shadow: 0 10px 30px rgba(0,0,0,0.7);";
            div.innerHTML = `
                <div style="display:flex; justify-content:space-between; align-items:center;">
                    <h2 style="margin:0; color:#00e676; letter-spacing:1px;">🛡️ FOCUS GUARD PRO</h2>
                    <div id="timer-badge" style="background:#333; padding:5px 12px; border-radius:20px; font-weight:bold; font-size:14px;">SESSION ACTIVE</div>
                </div>

                <div id="status-card" style="background:#1e1e1e; padding:20px; margin:20px 0; border-radius:12px; border-bottom: 4px solid #00e676; transition: all 0.3s ease;">
                    <div id="label" style="font-size:26px; font-weight:bold; margin-bottom:5px;">READY TO START</div>
                    <div id="stats" style="font-size:14px; color:#888;">Establishing neural link...</div>
                    <div style="background:#2c2c2c; height:8px; border-radius:10px; margin-top:15px; overflow:hidden;">
                        <div id="progress-bar" style="width:0%; height:100%; background:#00e676; transition: width 0.5s ease;"></div>
                    </div>
                </div>

                <div style="position:relative; overflow:hidden; border-radius:12px; border: 2px solid #333;">
                    <video id="vid" width="640" height="480" autoplay
                           style="display:block; transform: scaleX(-1); filter: contrast(1.1) brightness(0.9);"></video>
                </div>
            `;
            document.body.appendChild(div);

            // 2. Setup Sound Effect (Short Ping)
            window.alertSound = new Audio("https://actions.google.com/sounds/v1/alarms/beep_short.ogg");
            window.lastAlertTime = 0;

            const video = document.getElementById('vid');
            const stream = await navigator.mediaDevices.getUserMedia({video:true});
            video.srcObject = stream;
            await video.play();
        }
        setup();

        window.updateFocusUI = function(msg, stats, color, progress, playSound) {
            document.getElementById('label').innerText = msg;
            document.getElementById('label').style.color = color;
            document.getElementById('stats').innerText = stats;
            document.getElementById('status-card').style.borderBottomColor = color;
            document.getElementById('progress-bar').style.width = progress + "%";
            document.getElementById('progress-bar').style.backgroundColor = color;

            if (playSound && (Date.now() - window.lastAlertTime > 3000)) {
                window.alertSound.play();
                window.lastAlertTime = Date.now();
            }
        };
    ''')
    display(js)

# ==========================================
# 3. ENFORCEMENT ENGINE
# ==========================================
start_guard_ui()
study_goal = 60 # Seconds
current_focus = 0
time.sleep(2) # Warmup

print("System Online. Calibrating Gaze...")

try:
    while current_focus < study_goal:
        # 1. Sync Frame
        js_reply = eval_js('''
            (async () => {
                const video = document.getElementById('vid');
                const canvas = document.createElement('canvas');
                canvas.width = 640; canvas.height = 480;
                canvas.getContext('2d').drawImage(video, 0, 0);
                return canvas.toDataURL('image/jpeg', 0.8);
            })()
        ''')

        img_bytes = b64decode(js_reply.split(',')[1])
        img = cv2.imdecode(np.frombuffer(img_bytes, dtype=np.uint8), cv2.IMREAD_COLOR)
        # Fix: The AI sees the raw feed, so we flip the array before AI processing to match the UI
        img = cv2.flip(img, 1)
        mp_img = mp.Image(image_format=mp.ImageFormat.SRGB, data=cv2.cvtColor(img, cv2.COLOR_BGR2RGB))

        # 2. AI Inference
        face_res = face_landmarker.detect(mp_img)
        obj_res = obj_detector.detect(mp_img)

        # 3. Strict Distraction Checks
        phone = any(d.categories[0].category_name in ['cell phone', 'remote'] for d in obj_res.detections)
        eyes_closed = False
        looking_away = False

        if face_res.face_blendshapes:
            shapes = {b.category_name: b.score for b in face_res.face_blendshapes[0]}
            # Eye sensitivity (Lower = more sensitive)
            if shapes.get('eyeBlinkLeft', 0) > 0.45 and shapes.get('eyeBlinkRight', 0) > 0.45:
                eyes_closed = True

            # Head Movement / Gaze Accuracy
            nose = face_res.face_landmarks[0][1]
            if nose.x < 0.42 or nose.x > 0.58: # Narrowed detection window
                looking_away = True
        else:
            looking_away = True # Face not visible

        # 4. Action & UI Update
        progress = int((current_focus / study_goal) * 100)

        if phone:
            msg, color, sound = "📵 PHONE DETECTED", "#ff1744", True
        elif eyes_closed:
            msg, color, sound = "😴 WAKE UP!", "#ffea00", True
        elif looking_away:
            msg, color, sound = "👀 STAY ON TASK", "#ffea00", False
        else:
            current_focus += 1
            msg, color, sound = "🔥 DEEP FOCUS", "#00e676", False

        stats_str = f"Time: {current_focus}/{study_goal}s | Neural Confidence: 98%"
        eval_js(f"updateFocusUI('{msg}', '{stats_str}', '{color}', {progress}, {'true' if sound else 'false'})")

        time.sleep(1)

except Exception as e:
    print(f"Session Terminated: {e}")

eval_js("updateFocusUI('🎉 SESSION COMPLETE', 'Goal Reached: 60/60s', '#00e676', 100, false)")

<IPython.core.display.Javascript object>

System Online. Calibrating Gaze...


KeyboardInterrupt: 